# Test Bronze — MSPAS
Verifica las 3 tablas Bronze de MSPAS (Dropbox).  
Usa `limit(10)` — no ejecuta full scan.

In [0]:
from pyspark.sql import functions as F

BRONZE_SCHEMA = "sandbox_bronze_ss2"
_TABLES = [
    "dbx_primeras_causas_de_morbilidad_2015_a_2025",
    "dbx_morbilidad_enfermedades_cronicas_2015_a_2025",
    "dbx_morbilidad_grupo_materno_infantil_2012_a_2025",
]
# Columnas de auditoría que toda tabla Bronze MSPAS debe tener
_AUDIT_COLS    = ["ingestion_timestamp", "source_system", "source_file", "dropbox_path"]
_KEY_DATA_COLS = ["anio", "departamento", "municipio", "cie_10"]

---
## dbx_primeras_causas_de_morbilidad_2015_a_2025

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.dbx_primeras_causas_de_morbilidad_2015_a_2025")
df.printSchema()

In [0]:
display(df.limit(10))

---
## dbx_morbilidad_enfermedades_cronicas_2015_a_2025

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.dbx_morbilidad_enfermedades_cronicas_2015_a_2025")
df.printSchema()

In [0]:
display(df.limit(10))

---
## dbx_morbilidad_grupo_materno_infantil_2012_a_2025

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.dbx_morbilidad_grupo_materno_infantil_2012_a_2025")
df.printSchema()

In [0]:
display(df.limit(10))

---
## Chequeos de calidad (sin full scan)

In [0]:
# 1. Columnas de auditoría presentes en todas las tablas
print("── Columnas de auditoría Bronze presentes ──")
for t in _TABLES:
    cols = spark.table(f"{BRONZE_SCHEMA}.{t}").columns
    missing = [c for c in _AUDIT_COLS if c not in cols]
    print(f"  {t}: {'OK' if not missing else f'WARN — faltan: {missing}'}")

In [0]:
# 2. Columnas clave sin nulls en muestra
print("── Nulls en columnas clave (muestra 1k) ──")
for t in _TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(1000)
    cols_presentes = [c for c in _KEY_DATA_COLS if c in sample.columns]
    print(f"\n  {t}")
    for col in cols_presentes:
        nulls = sample.filter(F.col(col).isNull()).count()
        print(f"    {col} nulls: {nulls}")

In [0]:
# 3. Columnas de auditoría sin nulls
print("── Nulls en auditoría Bronze (muestra 1k) ──")
for t in _TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(1000)
    cols_presentes = [c for c in _AUDIT_COLS if c in sample.columns]
    print(f"\n  {t}")
    for col in cols_presentes:
        nulls = sample.filter(F.col(col).isNull()).count()
        status = "OK" if nulls == 0 else f"WARN — {nulls} nulls"
        print(f"    {col}: {status}")

In [0]:
# 4. Rango de años por tabla
print("── Rango de anio por tabla (muestra 5k) ──")
for t in _TABLES:
    result = (
        spark.table(f"{BRONZE_SCHEMA}.{t}")
        .limit(5000)
        .agg(F.min("anio").alias("min"), F.max("anio").alias("max"))
        .collect()[0]
    )
    print(f"  {t}: {result['min']} — {result['max']}")